In [18]:
# ============================================================
# SIMPLE EPL Transformer (Trajectory, NO-ODDS) — ROBUST + RESUMABLE
# - Auto-detect CSV (master vs base)
# - Robust date parsing
# - Optional extra stats used if present: shots/sot/corners/yellows/reds
# - TOKEN_COLS + CTX_COLS built dynamically and used everywhere consistently
# - SIMPLE model head: concat(home_vec, away_vec, ctx_vec) -> logits
#
# NOTE: CHANGE RUN_NAME whenever TOKEN_COLS / CTX_COLS changes!
# ============================================================

import os, math, random, glob, re
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# -------------------- CONFIG --------------------
CANDIDATES  = ["epl_master_train_full.csv", "epl_all_seasons_football_data.csv"]
MIN_DATE    = "2005-01-01"
NAN_THRESH  = 0.90
RANDOM_SEED = 42

SEQ_LEN     = 6
BATCH_SIZE  = 128
EPOCHS      = 10
LR          = 5e-4
WEIGHT_DECAY= 2e-2

D_MODEL     = 96
N_HEAD      = 4
N_LAYERS    = 2
D_FF        = 192
DROPOUT     = 0.2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- checkpointing ----
RUN_NAME = "epl_simple_v1"   # <<< change when features change
CKPT_ROOT = "checkpoints_epl_simple"
CKPT_PREFIX = "epl_simple"
CKPT_DIR = os.path.join(CKPT_ROOT, RUN_NAME)
os.makedirs(CKPT_DIR, exist_ok=True)

RESUME = True
SAVE_EVERY_N = 5
SAVE_KEEP_LAST_K = 3

fixtures = [
    ("2026-02-28", "Wolves",       "Aston Villa"),
    ("2026-02-28", "Bournemouth",  "Sunderland"),
    ("2026-02-28", "Newcastle",    "Everton"),
    ("2026-02-28", "Burnley",      "Brentford"),
    ("2026-02-28", "Liverpool",    "West Ham"),
    ("2026-02-28", "Leeds",        "Man City"),
]
fixtures_df = pd.DataFrame(fixtures, columns=["date","home_team","away_team"])
fixtures_df["date"] = pd.to_datetime(fixtures_df["date"])


# -------------------- helpers --------------------
def pick_col(df, *names):
    for n in names:
        if n in df.columns:
            return n
    return None

def clean_team_name(x: str) -> str:
    if pd.isna(x): return x
    s = str(x).strip()
    repl = {
        "Wolverhampton": "Wolves",
        "Manchester City": "Man City",
        "Manchester United": "Man United",
        "West Ham United": "West Ham",
        "Nott'm Forest": "Nottm Forest",
        "Nottingham Forest": "Nottm Forest",
        "Spurs": "Tottenham",
    }
    return repl.get(s, s)

def parse_date_series(df):
    if "date" in df.columns:
        return pd.to_datetime(df["date"], errors="coerce")
    if "Date" in df.columns:
        tmp = df["Date"].astype(str).str.replace(r"[^0-9/]", "", regex=True)
        dt = pd.to_datetime(tmp, format="%d/%m/%y", errors="coerce")
        bad = dt.isna()
        if bad.any():
            dt.loc[bad] = pd.to_datetime(tmp.loc[bad], format="%d/%m/%Y", errors="coerce")
        return dt
    return pd.to_datetime(pd.Series([pd.NaT]*len(df)))

def to_num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

def result_points(hg, ag):
    if hg > ag:  return 3, 0
    if hg < ag:  return 0, 3
    return 1, 1


# ============================================================
# 0) Load + cleanup (JOHN-style robust)
# ============================================================
CSV_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError(f"Couldn't find any of: {CANDIDATES}")

df0 = pd.read_csv(CSV_PATH, low_memory=False)

df0["date"] = parse_date_series(df0)
df0 = df0[df0["date"].notna()].copy()

home_team_col = pick_col(df0, "home_team", "HomeTeam")
away_team_col = pick_col(df0, "away_team", "AwayTeam")
hg_col        = pick_col(df0, "home_goals", "FTHG")
ag_col        = pick_col(df0, "away_goals", "FTAG")
target_col    = pick_col(df0, "target", "FTR", "ft_result")

if None in [home_team_col, away_team_col, hg_col, ag_col, target_col]:
    raise ValueError(
        "Missing required columns after auto-detect.\n"
        f"home_team_col={home_team_col}, away_team_col={away_team_col}, "
        f"hg_col={hg_col}, ag_col={ag_col}, target_col={target_col}\n"
        f"Found cols (sample): {list(df0.columns)[:40]}"
    )

df0 = df0.rename(columns={
    home_team_col: "home_team",
    away_team_col: "away_team",
    hg_col: "home_goals",
    ag_col: "away_goals",
    target_col: "target",
})

# drop ultra-missing cols
df0 = df0.loc[:, df0.isna().mean() <= NAN_THRESH].copy()

df0["home_team"] = df0["home_team"].map(clean_team_name)
df0["away_team"] = df0["away_team"].map(clean_team_name)

df0 = df0[df0["date"] >= pd.to_datetime(MIN_DATE)].copy()
df0["target"] = df0["target"].astype(str).str.upper().str.strip()
df0 = df0[df0["target"].isin(["H","D","A"])].copy()

to_num(df0, ["home_goals","away_goals","HS","AS","HST","AST","HC","AC","HY","AY","HR","AR"])
df0 = df0[df0["home_goals"].notna() & df0["away_goals"].notna()].copy()
df0 = df0.sort_values("date").reset_index(drop=True)

print(f"Loaded: {CSV_PATH}")
print("Rows after cleanup:", len(df0))
print("Date range:", df0["date"].min().date(), "->", df0["date"].max().date())


# ============================================================
# 1) Optional extra stats
# ============================================================
STAT_DEFS = [
    ("shots",           "HS",  "AS"),
    ("shots_on_target", "HST", "AST"),
    ("corners",         "HC",  "AC"),
    ("yellows",         "HY",  "AY"),
    ("reds",            "HR",  "AR"),
]
AVAILABLE_STATS = [name for name, h, a in STAT_DEFS if (h in df0.columns and a in df0.columns)]
print("Available extra stats:", AVAILABLE_STATS)


# ============================================================
# 2) Build long per-team history (sequences)
# ============================================================
rows = []
for i, r in df0.iterrows():
    hg, ag = float(r["home_goals"]), float(r["away_goals"])
    hp, ap = result_points(hg, ag)
    dt = r["date"]

    home_row = {
        "match_idx": i, "date": dt, "team": r["home_team"], "opp": r["away_team"],
        "is_home": 1.0, "gf": hg, "ga": ag, "gd": hg - ag, "pts": float(hp),
    }
    away_row = {
        "match_idx": i, "date": dt, "team": r["away_team"], "opp": r["home_team"],
        "is_home": 0.0, "gf": ag, "ga": hg, "gd": ag - hg, "pts": float(ap),
    }

    for name, hcol, acol in STAT_DEFS:
        if name in AVAILABLE_STATS:
            hv = r[hcol]; av = r[acol]
            home_row[name] = float(hv) if np.isfinite(hv) else np.nan
            away_row[name] = float(av) if np.isfinite(av) else np.nan

    rows.append(home_row)
    rows.append(away_row)

long = pd.DataFrame(rows).sort_values(["date","match_idx","team"]).reset_index(drop=True)

teams = sorted(pd.unique(pd.concat([df0["home_team"], df0["away_team"]], ignore_index=True)))
team_to_id = {t:i for i,t in enumerate(teams)}
long_by_team = {t: long[long["team"] == t].sort_values(["date","match_idx"]).reset_index(drop=True) for t in teams}

TOKEN_COLS = ["is_home","gf","ga","gd","pts"] + AVAILABLE_STATS


# ============================================================
# 3) Strictly causal pre-match context (CTX)
# ============================================================
points   = {t: 0.0 for t in teams}
gf_tot   = {t: 0.0 for t in teams}
ga_tot   = {t: 0.0 for t in teams}
last_date= {t: None for t in teams}

stat_tot = {name: {t: 0.0 for t in teams} for name in AVAILABLE_STATS}
stat_cnt = {name: {t: 0   for t in teams} for name in AVAILABLE_STATS}

ctx_rows = []
for i, r in df0.iterrows():
    dt = r["date"]
    ht = r["home_team"]; at = r["away_team"]
    hg = float(r["home_goals"]); ag = float(r["away_goals"])
    hp, ap = result_points(hg, ag)

    # ranking table pre-match
    table = []
    for t in teams:
        gd = gf_tot[t] - ga_tot[t]
        table.append((t, points[t], gd, gf_tot[t]))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    rank_map = {t: (rk+1) for rk, (t,_,_,_) in enumerate(table_sorted)}

    def rest_days(team):
        ld = last_date[team]
        if ld is None: return np.nan
        return float((dt - ld).days)

    row_ctx = {
        "match_idx": i,
        "home_pts_pre": float(points[ht]),
        "away_pts_pre": float(points[at]),
        "home_rank_pre": float(rank_map[ht]),
        "away_rank_pre": float(rank_map[at]),
        "home_rest_days": rest_days(ht),
        "away_rest_days": rest_days(at),
    }

    # pre-match stat means (season-to-date causal)
    for name, hcol, acol in STAT_DEFS:
        if name in AVAILABLE_STATS:
            def mean_stat(team):
                c = stat_cnt[name][team]
                return np.nan if c == 0 else float(stat_tot[name][team] / c)
            hmean = mean_stat(ht)
            amean = mean_stat(at)
            row_ctx[f"home_{name}_pre"] = hmean
            row_ctx[f"away_{name}_pre"] = amean
            row_ctx[f"{name}_pre_diff"] = hmean - amean

    ctx_rows.append(row_ctx)

    # update after match
    points[ht] += hp; points[at] += ap
    gf_tot[ht] += hg; ga_tot[ht] += ag
    gf_tot[at] += ag; ga_tot[at] += hg
    last_date[ht] = dt; last_date[at] = dt

    for name, hcol, acol in STAT_DEFS:
        if name in AVAILABLE_STATS:
            hv = r[hcol]; av = r[acol]
            if np.isfinite(hv):
                stat_tot[name][ht] += float(hv); stat_cnt[name][ht] += 1
            if np.isfinite(av):
                stat_tot[name][at] += float(av); stat_cnt[name][at] += 1

ctx = pd.DataFrame(ctx_rows).set_index("match_idx")
df = df0.join(ctx, how="left")


# ============================================================
# 4) Build supervised dataset rows
# ============================================================
def get_team_seq(team: str, match_idx: int):
    hist = long_by_team[team]
    past = hist[hist["match_idx"] < match_idx]
    if len(past) < SEQ_LEN:
        return None
    return past.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)

CTX_COLS = ["home_pts_pre","away_pts_pre","home_rank_pre","away_rank_pre","home_rest_days","away_rest_days"]
for name in AVAILABLE_STATS:
    CTX_COLS += [f"home_{name}_pre", f"away_{name}_pre", f"{name}_pre_diff"]

def build_dataset_rows():
    out = []
    for i, r in df.iterrows():
        ht, at = r["home_team"], r["away_team"]
        hs = get_team_seq(ht, i)
        asq = get_team_seq(at, i)
        if hs is None or asq is None:
            continue

        row = {
            "match_idx": i,
            "home_team": ht,
            "away_team": at,
            "home_seq": hs,
            "away_seq": asq,
            "target": r["target"],
        }
        row["match_idx"] = i
        for c in CTX_COLS:
            row[c] = r.get(c, np.nan)
        out.append(row)
    return pd.DataFrame(out)

data = build_dataset_rows().sort_values("match_idx").reset_index(drop=True)
print("Matches with enough history:", len(data), "out of", len(df0))

# label encode
le = LabelEncoder()
le.fit(["H","D","A"])  # force stable order
y_all = le.transform(data["target"].values)
class_names = list(le.classes_)
print("Classes:", class_names)

# time split
split_idx = int(len(data) * 0.85)
train_df = data.iloc[:split_idx].copy()
test_df  = data.iloc[split_idx:].copy()
y_train = le.transform(train_df["target"].values)
y_test  = le.transform(test_df["target"].values)

# ============================================================
# 5) Normalize using TRAIN stats (and impute ctx with TRAIN medians)
# ============================================================
token_stack = np.concatenate(train_df["home_seq"].tolist() + train_df["away_seq"].tolist(), axis=0)
tok_mean = token_stack.mean(axis=0)
tok_std  = token_stack.std(axis=0) + 1e-6

for c in CTX_COLS:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")
    test_df[c]  = pd.to_numeric(test_df[c], errors="coerce")
    med = float(np.nanmedian(train_df[c].to_numpy(dtype=float)))
    train_df[c] = train_df[c].fillna(med)
    test_df[c]  = test_df[c].fillna(med)

ctx_mat  = train_df[CTX_COLS].to_numpy(dtype=np.float32)
ctx_mean = ctx_mat.mean(axis=0)
ctx_std  = ctx_mat.std(axis=0) + 1e-6

def norm_tokens(x): return (x - tok_mean) / tok_std
def norm_ctx(x):    return (x - ctx_mean) / ctx_std


# ============================================================
# 6) Dataset / Dataloader
# ============================================================
class EPLSeqDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, y: np.ndarray):
        self.f = frame.reset_index(drop=True)
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.f)

    def __getitem__(self, idx):
        r = self.f.iloc[idx]
        home_seq = norm_tokens(r["home_seq"]).astype(np.float32)
        away_seq = norm_tokens(r["away_seq"]).astype(np.float32)
        ctxv = norm_ctx(r[CTX_COLS].to_numpy(dtype=np.float32)).astype(np.float32)
        hid = team_to_id[r["home_team"]]
        aid = team_to_id[r["away_team"]]
        return (
            torch.from_numpy(home_seq),
            torch.from_numpy(away_seq),
            torch.from_numpy(ctxv),
            torch.tensor(hid, dtype=torch.long),
            torch.tensor(aid, dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long),
        )

train_loader = DataLoader(EPLSeqDataset(train_df, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(EPLSeqDataset(test_df,  y_test),  batch_size=BATCH_SIZE, shuffle=False)


# ============================================================
# 7) SIMPLE MODEL (no collision features)
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        T = x.size(1)
        return x + self.pe[:, :T, :]

class SimpleTrajTransformer(nn.Module):
    def __init__(self, token_dim, ctx_dim, n_teams, n_classes):
        super().__init__()
        self.team_emb = nn.Embedding(n_teams, D_MODEL)
        self.in_proj  = nn.Linear(token_dim, D_MODEL)
        self.pos      = PositionalEncoding(D_MODEL, max_len=SEQ_LEN+1)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=D_FF,
            dropout=DROPOUT, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=N_LAYERS)

        self.cls = nn.Parameter(torch.zeros(1, 1, D_MODEL))
        nn.init.normal_(self.cls, std=0.02)

        self.ctx_mlp = nn.Sequential(
            nn.Linear(ctx_dim, D_MODEL),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(D_MODEL, D_MODEL),
        )

        # SIMPLE head: [home_vec, away_vec, ctx_vec]
        self.head = nn.Sequential(
            nn.Linear(D_MODEL*3, 256),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, n_classes)
        )

    def encode_team(self, seq, team_id):
        B, T, _ = seq.shape
        x = self.in_proj(seq)
        x = self.pos(x)
        cls = self.cls.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.team_emb(team_id).unsqueeze(1)
        z = self.encoder(x)
        return z[:, 0, :]  # CLS token

    def forward(self, home_seq, away_seq, ctx, home_id, away_id):
        h = self.encode_team(home_seq, home_id)
        a = self.encode_team(away_seq, away_id)
        c = self.ctx_mlp(ctx)
        feat = torch.cat([h, a, c], dim=1)
        return self.head(feat)

model = SimpleTrajTransformer(
    token_dim=len(TOKEN_COLS),
    ctx_dim=len(CTX_COLS),
    n_teams=len(teams),
    n_classes=len(class_names),
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


# ============================================================
# 8) Checkpointing
# ============================================================
def ckpt_path(epoch: int) -> str:
    return os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_epoch_{epoch:03d}.pt")

def find_latest_checkpoint():
    files = glob.glob(os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_epoch_*.pt"))
    if not files:
        return None
    def epnum(p):
        m = re.search(r"_epoch_(\d+)\.pt$", p)
        return int(m.group(1)) if m else -1
    return sorted(files, key=epnum)[-1]

def _cleanup_old_checkpoints(keep_last_k: int):
    if keep_last_k <= 0:
        return
    files = glob.glob(os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_epoch_*.pt"))
    if len(files) <= keep_last_k:
        return
    def epnum(p):
        m = re.search(r"_epoch_(\d+)\.pt$", p)
        return int(m.group(1)) if m else -1
    files = sorted(files, key=epnum)
    for p in files[:-keep_last_k]:
        try: os.remove(p)
        except OSError: pass

def save_checkpoint(epoch, best_test, best_state):
    payload = {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optim_state_dict": optim.state_dict(),
        "best_test": float(best_test),
        "best_state": best_state,
        "meta": {
            "RUN_NAME": RUN_NAME,
            "CSV_PATH": CSV_PATH,
            "SEQ_LEN": SEQ_LEN,
            "TOKEN_COLS": TOKEN_COLS,
            "CTX_COLS": CTX_COLS,
            "class_names": class_names,
            "team_to_id": team_to_id,
            "tok_mean": tok_mean, "tok_std": tok_std,
            "ctx_mean": ctx_mean, "ctx_std": ctx_std,
            "D_MODEL": D_MODEL, "N_HEAD": N_HEAD, "N_LAYERS": N_LAYERS,
            "D_FF": D_FF, "DROPOUT": DROPOUT,
            "AVAILABLE_STATS": AVAILABLE_STATS,
        }
    }
    torch.save(payload, ckpt_path(epoch))
    print(f"💾 Saved checkpoint: {ckpt_path(epoch)}")
    _cleanup_old_checkpoints(SAVE_KEEP_LAST_K)

def load_checkpoint(path: str):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optim.load_state_dict(ckpt["optim_state_dict"])
    start_epoch = int(ckpt["epoch"]) + 1
    best_test = float(ckpt.get("best_test", 1e9))
    best_state = ckpt.get("best_state", None)

    for state in optim.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)

    print(f"✅ Resumed from: {path}")
    print(f"   start_epoch={start_epoch}, best_test={best_test:.6f}")
    return start_epoch, best_test, best_state


# ============================================================
# 9) Train/Eval
# ============================================================
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    all_logits, all_y = [], []

    for home_seq, away_seq, ctxv, hid, aid, y in loader:
        home_seq = home_seq.to(DEVICE)
        away_seq = away_seq.to(DEVICE)
        ctxv = ctxv.to(DEVICE)
        hid = hid.to(DEVICE)
        aid = aid.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(home_seq, away_seq, ctxv, hid, aid)
        loss = criterion(logits, y)

        if train:
            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        total_loss += float(loss.item()) * y.size(0)
        all_logits.append(logits.detach().cpu())
        all_y.append(y.detach().cpu())

    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_y = torch.cat(all_y, dim=0).numpy()
    probs = torch.softmax(torch.from_numpy(all_logits), dim=1).numpy()
    return total_loss / len(loader.dataset), log_loss(all_y, probs)

start_epoch, best_test, best_state = 1, 1e9, None
if RESUME:
    latest = find_latest_checkpoint()
    if latest is not None:
        start_epoch, best_test, best_state = load_checkpoint(latest)
    else:
        print("ℹ️ No checkpoint found. Training from scratch.")

if best_state is None:
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

end_epoch = start_epoch + EPOCHS - 1
for ep in range(start_epoch, end_epoch + 1):
    tr_loss, tr_ll = run_epoch(train_loader, train=True)
    te_loss, te_ll = run_epoch(test_loader, train=False)
    print(f"Epoch {ep:03d} | train ll {tr_ll:.4f} | test ll {te_ll:.4f}")

    if te_ll < best_test:
        best_test = te_ll
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"⭐ New best test log loss: {best_test:.6f}")

    if (ep % SAVE_EVERY_N == 0) or (ep == end_epoch):
        save_checkpoint(ep, best_test, best_state)

model.load_state_dict(best_state)
print("✅ Best test log loss:", best_test)


# ============================================================
# 10) Temperature scaling (optional, but keeps interface consistent)
# ============================================================
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.logT = nn.Parameter(torch.zeros(()))

    def forward(self, logits):
        T = torch.exp(self.logT) + 1e-6
        return logits / T

scaler = TemperatureScaler().to(DEVICE)
optT = torch.optim.LBFGS(scaler.parameters(), lr=0.1, max_iter=50)

model.eval()
test_logits, test_y = [], []
with torch.no_grad():
    for home_seq, away_seq, ctxv, hid, aid, y in test_loader:
        logits = model(home_seq.to(DEVICE), away_seq.to(DEVICE), ctxv.to(DEVICE), hid.to(DEVICE), aid.to(DEVICE))
        test_logits.append(logits)
        test_y.append(y.to(DEVICE))
test_logits = torch.cat(test_logits, dim=0)
test_y = torch.cat(test_y, dim=0)

def closure():
    optT.zero_grad()
    loss = nn.CrossEntropyLoss()(scaler(test_logits), test_y)
    loss.backward()
    return loss

optT.step(closure)
T = float(torch.exp(scaler.logT).detach().cpu())
print("Calibrated Temperature T:", T)


# ============================================================
# 11) Predict fixtures (ctx length ALWAYS matches CTX_COLS)
# ============================================================
def latest_team_seq(team: str):
    team = clean_team_name(team)
    hist = long_by_team.get(team, None)
    if hist is None or len(hist) < SEQ_LEN:
        return None
    return hist.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)

# compute final rank at end of history
def compute_final_rank():
    table = []
    for t in teams:
        gd = float(gf_tot[t] - ga_tot[t])
        table.append((t, float(points[t]), gd, float(gf_tot[t])))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    return {t: (rk+1) for rk, (t,_,_,_) in enumerate(table_sorted)}

final_rank = compute_final_rank()

# precompute final stat means
final_stat_mean = {}
for name in AVAILABLE_STATS:
    final_stat_mean[name] = {}
    for t in teams:
        c = stat_cnt[name][t]
        final_stat_mean[name][t] = (stat_tot[name][t] / c) if c > 0 else np.nan

# train medians for ctx (for imputation)
ctx_train_meds = np.nanmedian(train_df[CTX_COLS].to_numpy(dtype=float), axis=0).astype(np.float32)

def fixture_ctx(dt, ht, at):
    ht = clean_team_name(ht); at = clean_team_name(at)

    def rest(team):
        ld = last_date.get(team, None)
        if ld is None:
            return np.nan
        return float((pd.to_datetime(dt) - ld).days)

    # build in the SAME ORDER as CTX_COLS
    ctx_vals = []
    base_map = {
        "home_pts_pre": float(points.get(ht, 0.0)),
        "away_pts_pre": float(points.get(at, 0.0)),
        "home_rank_pre": float(final_rank.get(ht, len(teams))),
        "away_rank_pre": float(final_rank.get(at, len(teams))),
        "home_rest_days": rest(ht),
        "away_rest_days": rest(at),
    }

    for c in CTX_COLS:
        if c in base_map:
            ctx_vals.append(base_map[c])
        elif c.startswith("home_") and c.endswith("_pre"):
            name = c[len("home_"):-len("_pre")]
            ctx_vals.append(float(final_stat_mean.get(name, {}).get(ht, np.nan)))
        elif c.startswith("away_") and c.endswith("_pre"):
            name = c[len("away_"):-len("_pre")]
            ctx_vals.append(float(final_stat_mean.get(name, {}).get(at, np.nan)))
        elif c.endswith("_pre_diff"):
            name = c[:-len("_pre_diff")]
            h = float(final_stat_mean.get(name, {}).get(ht, np.nan))
            a = float(final_stat_mean.get(name, {}).get(at, np.nan))
            ctx_vals.append(h - a)
        else:
            ctx_vals.append(np.nan)

    c = np.array(ctx_vals, dtype=np.float32)

    # impute missing using TRAIN medians (aligned index)
    miss = ~np.isfinite(c)
    if miss.any():
        c[miss] = ctx_train_meds[miss]

    # clip rest days if those cols exist
    if "home_rest_days" in CTX_COLS:
        j = CTX_COLS.index("home_rest_days")
        c[j] = np.clip(c[j], 0.0, 30.0)
    if "away_rest_days" in CTX_COLS:
        j = CTX_COLS.index("away_rest_days")
        c[j] = np.clip(c[j], 0.0, 30.0)

    return c

def predict_fixture(dt, ht, at):
    hs = latest_team_seq(ht)
    asq = latest_team_seq(at)
    if hs is None or asq is None:
        return None

    c = fixture_ctx(dt, ht, at)

    hs_t = torch.from_numpy(norm_tokens(hs)).unsqueeze(0).to(DEVICE)
    as_t = torch.from_numpy(norm_tokens(asq)).unsqueeze(0).to(DEVICE)
    c_t  = torch.from_numpy(norm_ctx(c)).unsqueeze(0).to(DEVICE)

    hid = torch.tensor([team_to_id[clean_team_name(ht)]], dtype=torch.long).to(DEVICE)
    aid = torch.tensor([team_to_id[clean_team_name(at)]], dtype=torch.long).to(DEVICE)

    model.eval()
    scaler.eval()
    with torch.no_grad():
        logits = model(hs_t, as_t, c_t, hid, aid)
        logits = scaler(logits)
        p = torch.softmax(logits, dim=1).cpu().numpy()[0]

    return dict(zip(class_names, p.tolist()))

rows_out = []
for _, r in fixtures_df.iterrows():
    dt, ht, at = r["date"], r["home_team"], r["away_team"]
    p = predict_fixture(dt, ht, at)
    if p is None:
        rows_out.append([dt, ht, at, np.nan, np.nan, np.nan, "NO DATA", np.nan])
        continue
    pH, pD, pA = p.get("H", np.nan), p.get("D", np.nan), p.get("A", np.nan)
    pick = max([("H",pH),("D",pD),("A",pA)], key=lambda x: x[1])[0]
    conf = float(max(pH, pD, pA))
    rows_out.append([dt, ht, at, pH, pD, pA, pick, conf])

summary = pd.DataFrame(rows_out, columns=["date","home_team","away_team","p_home","p_draw","p_away","pick","confidence"])
print("\n================= FIXTURE SUMMARY (Simple Transformer) =================")
print(summary.sort_values("confidence", ascending=False).reset_index(drop=True))

print("\n✅ Done.")

Loaded: epl_master_train_full.csv
Rows after cleanup: 7871
Date range: 2005-08-13 -> 2026-02-23
Available extra stats: ['shots', 'shots_on_target', 'corners', 'yellows', 'reds']
Matches with enough history: 7666 out of 7871
Classes: ['A', 'D', 'H']
✅ Resumed from: checkpoints_epl_simple/epl_simple_v1/epl_simple_epoch_010.pt
   start_epoch=11, best_test=1.006457


/tmp/ipykernel_4252/2392846942.py:526: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=DEVICE)


Epoch 011 | train ll 0.8926 | test ll 1.0433
Epoch 012 | train ll 0.8830 | test ll 1.0597
Epoch 013 | train ll 0.8731 | test ll 1.0655
Epoch 014 | train ll 0.8580 | test ll 1.0724
Epoch 015 | train ll 0.8458 | test ll 1.0941
💾 Saved checkpoint: checkpoints_epl_simple/epl_simple_v1/epl_simple_epoch_015.pt
Epoch 016 | train ll 0.8378 | test ll 1.0993
Epoch 017 | train ll 0.8197 | test ll 1.1088
Epoch 018 | train ll 0.8131 | test ll 1.1332
Epoch 019 | train ll 0.7910 | test ll 1.1542
Epoch 020 | train ll 0.7731 | test ll 1.1547
💾 Saved checkpoint: checkpoints_epl_simple/epl_simple_v1/epl_simple_epoch_020.pt
✅ Best test log loss: 1.006457249294818
Calibrated Temperature T: 1.101762056350708

================= FIXTURE SUMMARY (Simple Transformer) =================
        date    home_team    away_team    p_home    p_draw    p_away pick  \
0 2026-02-28    Liverpool     West Ham  0.678683  0.199639  0.121677    H   
1 2026-02-28        Leeds     Man City  0.160839  0.200484  0.638677    A   

In [19]:
# ============================================================
# Interactive "Next Game" UI — TRANSFORMER (Trajectory, NO-ODDS)
# FIXED: ctx vector now matches CTX_COLS length (e.g., 21)
# ============================================================

import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import torch

# -------------------- sanity checks --------------------
needed = [
    "df0","clean_team_name","long_by_team","TOKEN_COLS","SEQ_LEN",
    "team_to_id","class_names","tok_mean","tok_std","ctx_mean","ctx_std",
    "points","gf_tot","ga_tot","last_date","model","scaler","DEVICE","CTX_COLS"
]
_missing = [x for x in needed if x not in globals()]
if _missing:
    raise RuntimeError(f"Missing required variables from training cell: {_missing}")

def _is_finite(x):
    return x is not None and np.isfinite(x)

# -------------------- odds columns (optional) --------------------
ODDS_CANDIDATES = [
    ("odds_home","odds_draw","odds_away"),
    ("B365H","B365D","B365A"),
    ("PSH","PSD","PSA"),
    ("WHH","WHD","WHA"),
    ("VCH","VCD","VCA"),
]

def _find_odds_cols(df):
    for h, d, a in ODDS_CANDIDATES:
        if {h, d, a}.issubset(df.columns):
            return (h, d, a)
    return None

odds_cols = _find_odds_cols(df0)
if odds_cols is None:
    print("ℹ️ No odds columns found in df0. EV/edge disabled unless you type odds manually.")
else:
    print(f"✅ Using historical odds columns: {odds_cols}")

# Pre-build a cleaned df for odds lookup (fast)
if odds_cols is not None:
    hcol, dcol, acol = odds_cols
    dfx = df0.copy()
    dfx["date"] = pd.to_datetime(dfx["date"], errors="coerce")
    dfx["home_team"] = dfx["home_team"].map(clean_team_name)
    dfx["away_team"] = dfx["away_team"].map(clean_team_name)
    dfx[[hcol, dcol, acol]] = dfx[[hcol, dcol, acol]].apply(pd.to_numeric, errors="coerce")
    dfx = dfx.dropna(subset=["date","home_team","away_team"]).sort_values("date").reset_index(drop=True)
else:
    dfx = None

# -------------------- odds parsing helpers --------------------
def _parse_user_odds_to_decimal(s: str):
    s = (s or "").strip()
    if s == "":
        return None
    s2 = s.replace(" ", "")
    try:
        if s2.startswith("+") or s2.startswith("-"):
            a = float(s2)
            return 1.0 + (a/100.0) if a > 0 else 1.0 + (100.0/abs(a))

        x = float(s2)
        if 1.01 <= x <= 25.0:
            return x
        a = x
        return 1.0 + (a/100.0) if a > 0 else 1.0 + (100.0/abs(a))
    except Exception:
        return None

def implied_probs_from_decimal_odds(dec_h, dec_d, dec_a):
    if not (_is_finite(dec_h) and _is_finite(dec_d) and _is_finite(dec_a)):
        return None, None, None, None
    p_raw = np.array([1/dec_h, 1/dec_d, 1/dec_a], dtype=float)
    overround = float(p_raw.sum() - 1.0)
    p_fair = p_raw / p_raw.sum()
    return float(p_fair[0]), float(p_fair[1]), float(p_fair[2]), float(overround)

def get_historical_odds_decimal(home_team, away_team, n_last=8):
    if odds_cols is None or dfx is None:
        return None, None, None, "no_odds_columns"
    ht = clean_team_name(home_team)
    at = clean_team_name(away_team)

    def _med_triplet(sub):
        sub = sub[[hcol, dcol, acol]].dropna()
        if len(sub) == 0:
            return None
        m = sub.median()
        return float(m[hcol]), float(m[dcol]), float(m[acol])

    exact = dfx[(dfx["home_team"] == ht) & (dfx["away_team"] == at)].tail(n_last)
    t = _med_triplet(exact)
    if t is not None:
        return (*t, f"exact_fixture_median_last{min(n_last, len(exact))}")

    home_hist = dfx[dfx["home_team"] == ht].tail(n_last)
    away_hist = dfx[dfx["away_team"] == at].tail(n_last)
    t1 = _med_triplet(home_hist)
    t2 = _med_triplet(away_hist)

    if t1 is not None or t2 is not None:
        vals = []
        if t1 is not None: vals.append(t1)
        if t2 is not None: vals.append(t2)
        dec_h = float(np.median([v[0] for v in vals]))
        dec_d = float(np.median([v[1] for v in vals]))
        dec_a = float(np.median([v[2] for v in vals]))
        return dec_h, dec_d, dec_a, "team_level_median"

    league = dfx.tail(4000)
    t3 = _med_triplet(league)
    if t3 is not None:
        return (*t3, "league_median")
    return None, None, None, "no_history_found"

# -------------------- model helpers --------------------
def norm_tokens_np(x: np.ndarray) -> np.ndarray:
    return (x - tok_mean) / tok_std

def norm_ctx_np(x: np.ndarray) -> np.ndarray:
    return (x - ctx_mean) / ctx_std

def latest_team_seq(team: str):
    team = clean_team_name(team)
    hist = long_by_team.get(team, None)
    if hist is None or len(hist) < SEQ_LEN:
        return None
    return hist.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)

def compute_final_rank():
    teams_local = list(team_to_id.keys())
    table = []
    for t in teams_local:
        gd = float(gf_tot[t] - ga_tot[t])
        table.append((t, float(points[t]), gd, float(gf_tot[t])))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    return {t: (rk + 1) for rk, (t, _, _, _) in enumerate(table_sorted)}

final_rank = compute_final_rank()

# OPTIONAL: best practice is to define this in training:
# CTX_MED = train_df[CTX_COLS].median().to_dict()
CTX_MED = globals().get("CTX_MED", None)

REST_MED_HOME = float(globals().get("REST_MED_HOME", 7.0))
REST_MED_AWAY = float(globals().get("REST_MED_AWAY", 7.0))

# --- hard safety: CTX dims must match ---
if len(ctx_mean) != len(CTX_COLS) or len(ctx_std) != len(CTX_COLS):
    raise RuntimeError(
        f"CTX mismatch: len(CTX_COLS)={len(CTX_COLS)} but "
        f"len(ctx_mean)={len(ctx_mean)} len(ctx_std)={len(ctx_std)}. "
        "Your UI and model were built with different CTX_COLS. Re-run training cell or load matching checkpoint meta."
    )

# Prefer a numeric vector of medians aligned to CTX_COLS
# Best: set CTX_MED_VEC in training cell:
#   CTX_MED_VEC = np.nanmedian(train_df[CTX_COLS].to_numpy(dtype=float), axis=0).astype(np.float32)
CTX_MED_VEC = globals().get("CTX_MED_VEC", None)

def fixture_ctx_full(dt, ht, at):
    """Build ctx vector exactly matching CTX_COLS length."""
    ht = clean_team_name(ht); at = clean_team_name(at)
    dt = pd.to_datetime(dt)

    def rest(team):
        ld = last_date.get(team, None)
        if ld is None:
            return np.nan
        return float((dt - pd.to_datetime(ld)).days)

    # default fill (median vector is best)
    if CTX_MED_VEC is not None and len(CTX_MED_VEC) == len(CTX_COLS):
        c = CTX_MED_VEC.astype(np.float32).copy()
    else:
        # fallback: per-col dict median if provided; else use ctx_mean (raw) as last resort
        c = np.empty(len(CTX_COLS), dtype=np.float32)
        for i, cname in enumerate(CTX_COLS):
            if CTX_MED is not None and cname in CTX_MED and np.isfinite(CTX_MED[cname]):
                c[i] = float(CTX_MED[cname])
            else:
                c[i] = float(ctx_mean[i])  # raw mean -> normalizes to ~0

    # overrides for core known fields (only if they exist)
    core = {
        "home_pts_pre": float(points.get(ht, 0.0)),
        "away_pts_pre": float(points.get(at, 0.0)),
        "home_rank_pre": float(final_rank.get(ht, len(team_to_id))),
        "away_rank_pre": float(final_rank.get(at, len(team_to_id))),
        "home_rest_days": rest(ht),
        "away_rest_days": rest(at),
    }
    for k, v in core.items():
        if k in CTX_COLS:
            j = CTX_COLS.index(k)
            if (k.endswith("rest_days")) and (not np.isfinite(v)):
                v = REST_MED_HOME if "home_" in k else REST_MED_AWAY
            c[j] = float(v)

    # final NaN clean
    miss = ~np.isfinite(c)
    if miss.any():
        if CTX_MED_VEC is not None and len(CTX_MED_VEC) == len(CTX_COLS):
            c[miss] = CTX_MED_VEC[miss]
        else:
            # last resort: use ctx_mean raw values so normalized becomes ~0
            idx = np.where(miss)[0]
            for j in idx:
                c[j] = float(ctx_mean[j])

    return c


def predict_fixture_transformer(dt, ht, at):
    hs = latest_team_seq(ht)
    asq = latest_team_seq(at)
    if hs is None or asq is None:
        return None

    c = fixture_ctx_full(dt, ht, at)  # ✅ now matches CTX_COLS length

    hs_t = torch.from_numpy(norm_tokens_np(hs)).unsqueeze(0).to(DEVICE)
    as_t = torch.from_numpy(norm_tokens_np(asq)).unsqueeze(0).to(DEVICE)
    c_t  = torch.from_numpy(norm_ctx_np(c)).unsqueeze(0).to(DEVICE)

    hid = torch.tensor([team_to_id[clean_team_name(ht)]], dtype=torch.long).to(DEVICE)
    aid = torch.tensor([team_to_id[clean_team_name(at)]], dtype=torch.long).to(DEVICE)

    model.eval()
    if scaler is not None and hasattr(scaler, "eval"):
        scaler.eval()

    with torch.no_grad():
        logits = model(hs_t, as_t, c_t, hid, aid)
        logits = scaler(logits) if scaler is not None else logits
        p = torch.softmax(logits, dim=1).cpu().numpy()[0]

    return dict(zip(class_names, p.tolist()))

# -------------------- UI widgets --------------------
teams_ui = sorted(list(team_to_id.keys()))

date_picker = widgets.DatePicker(description="Date:", value=pd.Timestamp("2026-02-28").date())
home_dd = widgets.Dropdown(options=teams_ui, description="Home:",
                           value="Wolves" if "Wolves" in teams_ui else teams_ui[0],
                           layout=widgets.Layout(width="320px"))
away_dd = widgets.Dropdown(options=teams_ui, description="Away:",
                           value="Aston Villa" if "Aston Villa" in teams_ui else teams_ui[min(1, len(teams_ui)-1)],
                           layout=widgets.Layout(width="320px"))

odds_home = widgets.Text(description="Odds H:", value="", placeholder="e.g. +330 or 4.30",
                         layout=widgets.Layout(width="260px"))
odds_draw = widgets.Text(description="Odds D:", value="", placeholder="e.g. +260 or 3.60",
                         layout=widgets.Layout(width="260px"))
odds_away = widgets.Text(description="Odds A:", value="", placeholder="e.g. -120 or 1.83",
                         layout=widgets.Layout(width="260px"))

edge_thresh = widgets.FloatSlider(
    description="Edge ≥", min=0.0, max=0.15, step=0.005, value=0.03,
    readout_format=".3f", layout=widgets.Layout(width="420px")
)

btn = widgets.Button(description="Predict", button_style="primary", icon="check")
out = widgets.Output()

def on_click(_):
    with out:
        clear_output(wait=True)

        dt = date_picker.value
        ht = home_dd.value
        at = away_dd.value
        if ht == at:
            print("⚠️ Home and Away can’t be the same team.")
            return

        p = predict_fixture_transformer(dt, ht, at)
        if p is None:
            print("⚠️ Not enough team history for SEQ_LEN =", SEQ_LEN, "for one/both teams.")
            return

        probs = pd.Series({k: float(p.get(k, np.nan)) for k in ["H","D","A"]})
        pick = probs.idxmax()
        conf = float(probs.max())

        print(f"{pd.to_datetime(dt).date()} | {clean_team_name(ht)} vs {clean_team_name(at)}")
        print(f"Model pick: {pick}  (confidence={conf:.3f})")

        display(pd.DataFrame(
            [["Home (H)", probs["H"]], ["Draw (D)", probs["D"]], ["Away (A)", probs["A"]]],
            columns=["Outcome","p_model"]
        ).style.format({"p_model":"{:.3f}"}))

        # odds -> decimal; fill from history if missing
        user_dec_h = _parse_user_odds_to_decimal(odds_home.value)
        user_dec_d = _parse_user_odds_to_decimal(odds_draw.value)
        user_dec_a = _parse_user_odds_to_decimal(odds_away.value)

        need_fill = (user_dec_h is None) or (user_dec_d is None) or (user_dec_a is None)
        hist_src = None
        if need_fill:
            hist_dec_h, hist_dec_d, hist_dec_a, hist_src = get_historical_odds_decimal(ht, at)
        else:
            hist_dec_h = hist_dec_d = hist_dec_a = None

        dec_h = user_dec_h if user_dec_h is not None else hist_dec_h
        dec_d = user_dec_d if user_dec_d is not None else hist_dec_d
        dec_a = user_dec_a if user_dec_a is not None else hist_dec_a

        if not (_is_finite(dec_h) and _is_finite(dec_d) and _is_finite(dec_a)):
            print("Odds not provided / could not be filled → EV/edge not computed (model probs only).")
            return

        pbh, pbd, pba, overround = implied_probs_from_decimal_odds(dec_h, dec_d, dec_a)
        if hist_src is not None and need_fill:
            print(f"Odds filled from history: {hist_src}")
        print(f"Odds used (decimal): H={dec_h:.3f}  D={dec_d:.3f}  A={dec_a:.3f}")
        print(f"Overround (book margin): {overround:.3f}")

        rows = []
        for lab, name, dec, p_book in [
            ("H","Home (H)",dec_h,pbh),
            ("D","Draw (D)",dec_d,pbd),
            ("A","Away (A)",dec_a,pba),
        ]:
            p_model = float(probs[lab])
            edge    = p_model - float(p_book)
            ev      = (p_model * float(dec)) - 1.0
            rows.append([name, p_model, float(p_book), edge, ev])

        table = pd.DataFrame(rows, columns=["Outcome","p_model","p_book","edge","EV"])
        best = table.loc[table["EV"].idxmax()]
        do_bet = (float(best["EV"]) > 0) and (float(best["edge"]) >= float(edge_thresh.value))

        print(f"Best EV side: {best['Outcome']} | EV={best['EV']:.3f} | edge={best['edge']:.3f}")
        print("Suggestion:", "BET ✅" if do_bet else "NO BET ❌")

        display(table.style.format({"p_model":"{:.3f}","p_book":"{:.3f}","edge":"{:.3f}","EV":"{:.3f}"}))

btn.on_click(on_click)

ui1 = widgets.HBox([date_picker, home_dd, away_dd])
ui2 = widgets.HBox([odds_home, odds_draw, odds_away])
ui3 = widgets.HBox([edge_thresh, btn])

display(ui1, ui2, ui3, out)
print("✅ Transformer UI ready. CTX length =", len(CTX_COLS), "| Leave odds blank to auto-fill (if available).")

✅ Using historical odds columns: ('odds_home', 'odds_draw', 'odds_away')


Output()

✅ Transformer UI ready. CTX length = 21 | Leave odds blank to auto-fill (if available).


In [21]:
# ============================================================
# EPL BANKROLL SIMULATOR (KATHY MODEL) — FULL SCRIPT
# Works when:
#   - data has match_idx (it does)
#   - data has NO date (your case)
# Fix:
#   - recreate df0.match_idx by reproducing the same df0 ordering used in training
# ============================================================

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from ipywidgets import (
    FloatSlider, FloatText, Dropdown, Checkbox,
    VBox, HBox, Output, interactive_output
)
from IPython.display import display

# =========================
# 0) Kathy objects must exist
# =========================
needed = ["df0","data","model","DEVICE","class_names","CTX_COLS","team_to_id","clean_team_name","norm_tokens","norm_ctx"]
miss = [x for x in needed if x not in globals()]
if miss:
    raise RuntimeError(f"Missing required objects in notebook: {miss}")

scaler_local = globals().get("scaler", None)

KATHY = {
    "df0": df0,
    "data": data,
    "model": model,
    "scaler": scaler_local,     # can be None
    "DEVICE": DEVICE,
    "class_names": class_names,
    "CTX_COLS": CTX_COLS,
    "team_to_id": team_to_id,
    "clean_team_name": clean_team_name,
    "norm_tokens": norm_tokens,
    "norm_ctx": norm_ctx,
}

ODDS_CANDIDATES = [
    ("odds_home", "odds_draw", "odds_away"),
    ("B365H", "B365D", "B365A"),
    ("PSH", "PSD", "PSA"),
    ("WHH", "WHD", "WHA"),
    ("VCH", "VCD", "VCA"),
]

def find_odds_triplet(df: pd.DataFrame):
    for h, d, a in ODDS_CANDIDATES:
        if {h, d, a}.issubset(df.columns):
            return (h, d, a)
    return None

def _get_realized_result_col(df0: pd.DataFrame) -> str:
    for c in ["target", "ft_result", "FTR"]:
        if c in df0.columns:
            return c
    raise ValueError("df0 needs realized result column: one of ['target','ft_result','FTR'].")

def implied_probs_from_decimal_odds(dec_h, dec_d, dec_a):
    if not (np.isfinite(dec_h) and np.isfinite(dec_d) and np.isfinite(dec_a)):
        return None
    p_raw = np.array([1/dec_h, 1/dec_d, 1/dec_a], dtype=float)
    overround = float(p_raw.sum() - 1.0)
    p_fair = p_raw / p_raw.sum()
    return float(p_fair[0]), float(p_fair[1]), float(p_fair[2]), overround

def model_probs_for_batch(model, scaler, home_seq, away_seq, ctx, home_id, away_id):
    model.eval()
    if scaler is not None and hasattr(scaler, "eval"):
        scaler.eval()
    with torch.no_grad():
        logits = model(home_seq, away_seq, ctx, home_id, away_id)
        logits = scaler(logits) if scaler is not None else logits
        return torch.softmax(logits, dim=1)

def rebuild_df0_match_idx_like_training(df0_in: pd.DataFrame) -> pd.DataFrame:
    """
    Rebuild df0['match_idx'] to match Kathy training:
      df0['date'] parsed -> drop NaT -> sort by date -> reset_index(drop=True)
      then match_idx = 0..N-1
    This works because Kathy created match_idx implicitly as i in df0.iterrows()
    after df0 was sorted by date and reset_index.
    """
    df = df0_in.copy()

    # date parse
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"].notna()].copy()

    # IMPORTANT: mimic training ordering
    df = df.sort_values("date").reset_index(drop=True)

    # add match_idx
    df["match_idx"] = np.arange(len(df), dtype=int)
    return df

def build_scored_table(cfg):
    df0 = cfg["df0"]
    data = cfg["data"].copy()
    model = cfg["model"]
    scaler = cfg["scaler"]
    DEVICE = cfg["DEVICE"]
    class_names = list(cfg["class_names"])
    CTX_COLS = list(cfg["CTX_COLS"])
    team_to_id = cfg["team_to_id"]
    clean_team_name = cfg["clean_team_name"]
    norm_tokens = cfg["norm_tokens"]
    norm_ctx = cfg["norm_ctx"]

    # must have match_idx in data
    if "match_idx" not in data.columns:
        raise ValueError("data has no 'match_idx'. Rebuild data to include it.")

    # rebuild df0.match_idx if missing
    if "match_idx" not in df0.columns:
        df0 = rebuild_df0_match_idx_like_training(df0)
        print("✅ Rebuilt df0['match_idx'] by sorting df0 by date (to match training).")
    else:
        df0 = df0.copy()
        df0["match_idx"] = pd.to_numeric(df0["match_idx"], errors="coerce").astype(int)

    # odds
    odds_cols = find_odds_triplet(df0)
    if odds_cols is None:
        raise ValueError(f"No odds columns found in df0. Tried: {ODDS_CANDIDATES}")
    HODD, DODD, AODD = odds_cols
    print("✅ Using odds columns:", odds_cols)

    # realized result col
    res_col = _get_realized_result_col(df0)

    # clean df0 columns
    df0["date"] = pd.to_datetime(df0["date"], errors="coerce")
    df0[res_col] = df0[res_col].astype(str).str.strip().str.upper()
    df0[[HODD, DODD, AODD]] = df0[[HODD, DODD, AODD]].apply(pd.to_numeric, errors="coerce")

    # ensure labels include H/D/A
    label_to_i = {lab: i for i, lab in enumerate(class_names)}
    for lab in ["H","D","A"]:
        if lab not in label_to_i:
            raise ValueError(f"class_names must include H/D/A. Got {class_names}")

    # join meta onto data by match_idx
    meta = df0[["match_idx","date",res_col,HODD,DODD,AODD]].rename(columns={res_col:"ft_result"}).copy()
    sim = data.merge(meta, on="match_idx", how="left")

    # require
    need = ["date","ft_result",HODD,DODD,AODD,"home_team","away_team","home_seq","away_seq"] + CTX_COLS
    miss = [c for c in need if c not in sim.columns]
    if miss:
        raise ValueError(f"Aligned sim table missing columns: {miss}")

    # filter usable rows
    sim["date"] = pd.to_datetime(sim["date"], errors="coerce")
    sim["ft_result"] = sim["ft_result"].astype(str).str.strip().str.upper()
    sim = sim.dropna(subset=["date",HODD,DODD,AODD]).copy()
    sim = sim[sim["ft_result"].isin(["H","D","A"])].copy()

    # team ids
    def _map_team_id(x):
        k = clean_team_name(x)
        if k not in team_to_id:
            raise KeyError(f"Team '{x}' -> '{k}' not found in team_to_id.")
        return team_to_id[k]

    sim["home_id"] = sim["home_team"].map(_map_team_id)
    sim["away_id"] = sim["away_team"].map(_map_team_id)

    # tensors
    home_seq_arr = np.stack([norm_tokens(x).astype(np.float32) for x in sim["home_seq"].values], axis=0)
    away_seq_arr = np.stack([norm_tokens(x).astype(np.float32) for x in sim["away_seq"].values], axis=0)
    ctx_mat = sim[CTX_COLS].to_numpy(dtype=np.float32)
    ctx_arr = norm_ctx(ctx_mat).astype(np.float32)

    home_seq_t = torch.from_numpy(home_seq_arr).to(DEVICE)
    away_seq_t = torch.from_numpy(away_seq_arr).to(DEVICE)
    ctx_t      = torch.from_numpy(ctx_arr).to(DEVICE)
    home_id_t  = torch.from_numpy(sim["home_id"].to_numpy(np.int64)).to(DEVICE)
    away_id_t  = torch.from_numpy(sim["away_id"].to_numpy(np.int64)).to(DEVICE)

    probs = model_probs_for_batch(model, scaler, home_seq_t, away_seq_t, ctx_t, home_id_t, away_id_t).detach().cpu().numpy()
    sim["pH"] = probs[:, label_to_i["H"]]
    sim["pD"] = probs[:, label_to_i["D"]]
    sim["pA"] = probs[:, label_to_i["A"]]

    # book fair probs + overround
    fair = sim.apply(lambda r: implied_probs_from_decimal_odds(float(r[HODD]), float(r[DODD]), float(r[AODD])), axis=1)
    fair = pd.DataFrame(fair.tolist(), columns=["p_book_H","p_book_D","p_book_A","overround"], index=sim.index)
    sim = pd.concat([sim, fair], axis=1)

    # EV + edge
    sim["EV_H"] = sim["pH"] * sim[HODD] - 1.0
    sim["EV_D"] = sim["pD"] * sim[DODD] - 1.0
    sim["EV_A"] = sim["pA"] * sim[AODD] - 1.0

    sim["edge_H"] = sim["pH"] - sim["p_book_H"]
    sim["edge_D"] = sim["pD"] - sim["p_book_D"]
    sim["edge_A"] = sim["pA"] - sim["p_book_A"]

    sim = sim.sort_values("date").reset_index(drop=True)

    # quick alignment sanity print
    hit = sim["ft_result"].notna().mean()
    print(f"🔎 Joined ft_result coverage: {hit*100:.1f}% (after filtering rows with odds/date)")
    return sim, (HODD, DODD, AODD)

# ===== strategy helpers =====
def _kelly_fraction(p, dec_odds):
    b = dec_odds - 1.0
    if b <= 0: return 0.0
    return float(max(0.0, (p * dec_odds - 1.0) / b))

def _choose_side_max_ev(row, allow_draw=True):
    evs = {"H": row["EV_H"], "D": row["EV_D"], "A": row["EV_A"]} if allow_draw else {"H": row["EV_H"], "A": row["EV_A"]}
    return max(evs.items(), key=lambda kv: kv[1])[0]

def run_strategy(
    sim_all: pd.DataFrame,
    odds_cols,
    start_date="2005-01-01",
    end_date=None,
    bankroll0=200.0,
    stake_method="Fractional Kelly",
    flat_stake=5.0,
    kelly_fraction=0.25,
    max_stake_pct=0.03,
    conf_thresh=0.45,
    ev_thresh=0.01,
    edge_thresh=0.02,
    max_odds=4.5,
    min_odds=1.30,
    max_overround=0.08,
    allow_draw=True,
    use_odds_cap=True
):
    HODD, DODD, AODD = odds_cols
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    sim = sim_all.copy()
    sim = sim[(sim["date"] >= pd.to_datetime(start_date)) & (sim["date"] <= pd.to_datetime(end_date))].copy()

    bankroll = float(bankroll0)
    bankroll_path, bet_amounts, bet_sides, do_bets, profits = [], [], [], [], []

    for _, r in sim.iterrows():
        side = _choose_side_max_ev(r, allow_draw=allow_draw)
        p = float(r["pH"] if side=="H" else (r["pD"] if side=="D" else r["pA"]))
        dec = float(r[HODD] if side=="H" else (r[DODD] if side=="D" else r[AODD]))
        edge = float(r["edge_H"] if side=="H" else (r["edge_D"] if side=="D" else r["edge_A"]))
        ev = float(r["EV_H"] if side=="H" else (r["EV_D"] if side=="D" else r["EV_A"]))
        conf = float(max(r["pH"], r["pD"], r["pA"]))

        ok = True
        if (not np.isfinite(r["overround"])) or (r["overround"] > float(max_overround)): ok = False
        if ev < float(ev_thresh): ok = False
        if edge < float(edge_thresh): ok = False
        if conf < float(conf_thresh): ok = False
        if use_odds_cap and (dec > float(max_odds) or dec < float(min_odds)): ok = False

        if (not ok) or bankroll <= 0:
            do_bets.append(False); bet_amounts.append(0.0); bet_sides.append(side)
            profits.append(0.0); bankroll_path.append(bankroll)
            continue

        if stake_method == "Flat":
            stake = float(flat_stake)
        else:
            f = _kelly_fraction(p, dec)
            stake = bankroll * float(kelly_fraction) * f

        cap = bankroll * float(max_stake_pct)
        stake = float(np.clip(stake, 0.0, cap))

        if stake < 0.50:
            do_bets.append(False); bet_amounts.append(0.0); bet_sides.append(side)
            profits.append(0.0); bankroll_path.append(bankroll)
            continue

        win = (r["ft_result"] == side)
        profit = stake * (dec - 1.0) if win else -stake
        bankroll += profit

        do_bets.append(True); bet_amounts.append(stake); bet_sides.append(side)
        profits.append(profit); bankroll_path.append(bankroll)

    sim = sim.reset_index(drop=True)
    sim["bet_side"] = bet_sides
    sim["do_bet"] = do_bets
    sim["stake"] = bet_amounts
    sim["profit"] = profits
    sim["bankroll"] = bankroll_path
    sim["cum_profit"] = sim["profit"].cumsum()

    n_bets = int(sim["do_bet"].sum())
    total_staked = float(sim.loc[sim["do_bet"], "stake"].sum())
    net_profit = float(sim["cum_profit"].iloc[-1]) if len(sim) else 0.0
    roi = (net_profit / total_staked) if total_staked > 0 else 0.0

    summary = {
        "bets": n_bets,
        "total_staked": total_staked,
        "net_profit": net_profit,
        "ROI_on_staked": roi,
        "final_bankroll": float(sim["bankroll"].iloc[-1]) if len(sim) else bankroll0,
        "avg_stake": float(sim.loc[sim["do_bet"], "stake"].mean()) if n_bets else 0.0,
        "odds_cols": odds_cols,
    }
    return summary, sim

def plot_results(sim, title):
    fig, ax = plt.subplots(figsize=(12, 5), dpi=200)
    ax.plot(sim["date"], sim["bankroll"], linewidth=2.5)
    ax.axhline(sim["bankroll"].iloc[0], linestyle="--", linewidth=1.2)
    ax.set_title(title, pad=12)
    ax.set_xlabel("Date")
    ax.set_ylabel("Bankroll ($)")
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

# ============================================================
# Build scored table
# ============================================================
SIM_ALL, odds_cols = build_scored_table(KATHY)
print("✅ Scored rows:", len(SIM_ALL))

# ============================================================
# Interactive dashboard
# ============================================================
out = Output()

def dashboard(
    bankroll0,
    stake_method,
    flat_stake,
    kelly_fraction,
    max_stake_pct,
    conf_thresh,
    ev_thresh,
    edge_thresh,
    min_odds,
    max_odds,
    max_overround,
    allow_draw,
    use_odds_cap
):
    with out:
        out.clear_output(wait=True)
        summ, sim = run_strategy(
            SIM_ALL,
            odds_cols=odds_cols,
            start_date="2005-01-01",
            end_date=None,
            bankroll0=bankroll0,
            stake_method=stake_method,
            flat_stake=flat_stake,
            kelly_fraction=kelly_fraction,
            max_stake_pct=max_stake_pct,
            conf_thresh=conf_thresh,
            ev_thresh=ev_thresh,
            edge_thresh=edge_thresh,
            min_odds=min_odds,
            max_odds=max_odds,
            max_overround=max_overround,
            allow_draw=allow_draw,
            use_odds_cap=use_odds_cap
        )
        print(pd.Series(summ).to_string())
        title = f"Bankroll | {stake_method} | conf≥{conf_thresh:.2f}, EV≥{ev_thresh:.2%}, edge≥{edge_thresh:.2%}"
        plot_results(sim, title)

controls = {
    "bankroll0": FloatText(value=200.0, description="Start $"),
    "stake_method": Dropdown(options=["Fractional Kelly", "Flat"], value="Fractional Kelly", description="Stake"),
    "flat_stake": FloatSlider(value=5.0, min=1.0, max=50.0, step=1.0, description="Flat $"),
    "kelly_fraction": FloatSlider(value=0.25, min=0.05, max=0.50, step=0.05, description="Kelly frac"),
    "max_stake_pct": FloatSlider(value=0.03, min=0.01, max=0.10, step=0.01, description="Max %/bet"),
    "conf_thresh": FloatSlider(value=0.45, min=0.33, max=0.65, step=0.01, description="Conf min"),
    "ev_thresh": FloatSlider(value=0.01, min=0.00, max=0.05, step=0.005, description="EV min"),
    "edge_thresh": FloatSlider(value=0.02, min=0.00, max=0.06, step=0.005, description="Edge min"),
    "min_odds": FloatSlider(value=1.30, min=1.01, max=2.50, step=0.05, description="Min odds"),
    "max_odds": FloatSlider(value=4.50, min=2.00, max=15.0, step=0.5, description="Max odds"),
    "max_overround": FloatSlider(value=0.08, min=0.00, max=0.15, step=0.01, description="Max vig"),
    "allow_draw": Checkbox(value=True, description="Allow draws"),
    "use_odds_cap": Checkbox(value=True, description="Use odds caps"),
}

ui = VBox([
    HBox([controls["bankroll0"], controls["stake_method"]]),
    HBox([controls["flat_stake"], controls["kelly_fraction"], controls["max_stake_pct"]]),
    HBox([controls["conf_thresh"], controls["ev_thresh"], controls["edge_thresh"]]),
    HBox([controls["min_odds"], controls["max_odds"], controls["max_overround"]]),
    HBox([controls["allow_draw"], controls["use_odds_cap"]]),
])

display(ui, out)
interactive_output(dashboard, controls)

✅ Rebuilt df0['match_idx'] by sorting df0 by date (to match training).
✅ Using odds columns: ('odds_home', 'odds_draw', 'odds_away')
🔎 Joined ft_result coverage: 100.0% (after filtering rows with odds/date)
✅ Scored rows: 7666


Output()

Output()